In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import anndata as ad
import scanpy as sc

import sys
sys.path.append('/home/jwinchel/Heimdall/')

from Heimdall.utils import convert_to_ensembl_ids

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:

In [6]:
# read in common genes
gene_overlap = pd.read_csv('/compute/oven-0-13/jwinchel/gene_overlap.csv')
common_genes = list(sorted(gene_overlap['ensembl_gene'].unique()))
print(len(common_genes))

958


In [3]:
import os, re

# get spatial transcriptomics files
data_dir = '/compute/oven-0-13/jwinchel/'
st_files = sorted([os.path.join(data_dir,x,x+'-Flat_files_and_images/',x+"_exprMat_file.csv") for x in os.listdir(data_dir) if x.startswith('Lung')])
st_files

['/compute/oven-0-13/jwinchel/Lung12/Lung12-Flat_files_and_images/Lung12_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung13/Lung13-Flat_files_and_images/Lung13_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung5_Rep1/Lung5_Rep1-Flat_files_and_images/Lung5_Rep1_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung5_Rep2/Lung5_Rep2-Flat_files_and_images/Lung5_Rep2_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung5_Rep3/Lung5_Rep3-Flat_files_and_images/Lung5_Rep3_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung6/Lung6-Flat_files_and_images/Lung6_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung9_Rep1/Lung9_Rep1-Flat_files_and_images/Lung9_Rep1_exprMat_file.csv',
 '/compute/oven-0-13/jwinchel/Lung9_Rep2/Lung9_Rep2-Flat_files_and_images/Lung9_Rep2_exprMat_file.csv']

In [ ]:
# st_data = ad.concat([ad.read_csv(f) for f in st_files], join='outer')
ad_list = []
for i,f in enumerate(st_files):
    st_sample = ad.read_csv(f)
    
    # move fov, cell_ID to obs
    st_sample.obs[['fov', 'cell_ID']] = st_sample.X[:,:2]
    st_sample = st_sample[:,~st_sample.var_names.isin(['fov','cell_ID'])]
    
    # convert genes to ensembl IDs 
    st_sample,_ = convert_to_ensembl_ids(st_sample, '/compute/oven-0-13/jwinchel/')
    
    # restrict to common genes
    st_sample = st_sample[:,common_genes]
    st_sample.obs['tissue'] = f.split('/')[4]
    ad_list.append(st_sample)
del st_sample
st_data = ad.concat(ad_list, join='outer')
del ad_list

/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)


Index(['fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1',
       'ACKR3', 'ACKR4',
       ...
       'NegPrb14', 'NegPrb15', 'NegPrb16', 'NegPrb17', 'NegPrb18', 'NegPrb19',
       'NegPrb20', 'NegPrb21', 'NegPrb22', 'NegPrb23'],
      dtype='object', length=982)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full
/tmp/ipykernel_681212/3591160653.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_sample.obs['tissue'] = f.split('/')[4]
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)


Index(['fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1',
       'ACKR3', 'ACKR4',
       ...
       'NegPrb14', 'NegPrb15', 'NegPrb16', 'NegPrb17', 'NegPrb18', 'NegPrb19',
       'NegPrb20', 'NegPrb21', 'NegPrb22', 'NegPrb23'],
      dtype='object', length=982)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full
/tmp/ipykernel_681212/3591160653.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_sample.obs['tissue'] = f.split('/')[4]
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)


Index(['fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1',
       'ACKR3', 'ACKR4',
       ...
       'NegPrb14', 'NegPrb15', 'NegPrb16', 'NegPrb17', 'NegPrb18', 'NegPrb19',
       'NegPrb20', 'NegPrb21', 'NegPrb22', 'NegPrb23'],
      dtype='object', length=982)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


... (7 outputs left out)

Index(['fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1',
       'ACKR3', 'ACKR4',
       ...
       'NegPrb14', 'NegPrb15', 'NegPrb16', 'NegPrb17', 'NegPrb18', 'NegPrb19',
       'NegPrb20', 'NegPrb21', 'NegPrb22', 'NegPrb23'],
      dtype='object', length=982)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full
/tmp/ipykernel_681212/3591160653.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_sample.obs['tissue'] = f.split('/')[4]
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)


Index(['fov', 'cell_ID', 'AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1',
       'ACKR3', 'ACKR4',
       ...
       'NegPrb14', 'NegPrb15', 'NegPrb16', 'NegPrb17', 'NegPrb18', 'NegPrb19',
       'NegPrb20', 'NegPrb21', 'NegPrb22', 'NegPrb23'],
      dtype='object', length=982)
human
/compute/oven-0-13/jwinchel
Mapping data directory: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human
Loading mapping from cache: /compute/oven-0-13/jwinchel/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 957 out of 980 identifiers (97.7%)


/home/jwinchel/Heimdall/Heimdall/utils.py:457: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns["gene_mapping:symbol_to_ensembl"] = gene_mapping.mapping_full
/tmp/ipykernel_681212/3591160653.py:15: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_sample.obs['tissue'] = f.split('/')[4]
/home/jwinchel/miniconda3/envs/heimdall/lib/python3.10/site-packages/anndata/_core/anndata.py:1757: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [51]:
st_data

AnnData object with n_obs × n_vars = 800559 × 958
    obs: 'fov', 'cell_ID', 'tissue', 'sdimx', 'sdimy', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'cell_ID_short'

In [34]:
st_data.obs['tissue'].unique()

array(['Lung12', 'Lung13', 'Lung5_Rep1', 'Lung5_Rep2', 'Lung5_Rep3',
       'Lung6', 'Lung9_Rep1', 'Lung9_Rep2'], dtype=object)

In [35]:
metadata = pd.read_csv('/compute/oven-0-13/jwinchel/NSCLC_ST_cell_metadata.csv')
metadata[['Run_Tissue_name','fov','cell_ID']].drop_duplicates()

metadata['cell_ID_short'] =metadata['cell_ID'].apply(lambda x: x.split('_')[-1]).astype(int)
# metadata[['Slide_name','fov','cell_ID']]

In [36]:
metadata['fov'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [38]:
st_data.obs = st_data.obs.merge(metadata.drop(columns=['tissue','cell_ID']),left_on=['tissue','fov','cell_ID'],right_on=['Run_Tissue_name','fov','cell_ID_short'],how='left')

In [39]:
cat_cols = [c for c in st_data.obs.columns if st_data.obs[c].dtype in ['category', 'O']]
print(cat_cols)
st_data.obs[cat_cols] = st_data.obs[cat_cols].astype(str)

['tissue', 'dualfiles', 'Slide_name', 'Run_name', 'ISH.concentration', 'Dash', 'Run_Tissue_name', 'Panel', 'remove_flagged_cells', 'patient', 'cell_type', 'niche']


In [40]:
cat_cols = [c for c in st_data.obs.columns if st_data.obs[c].dtype in ['category', 'O']]
st_data.obs['remove_flagged_cells'].dtype

dtype('O')

In [50]:
st_data.write_h5ad('/compute/oven-0-13/jwinchel/NSCLC_ST_with_unlabeled.h5ad')

st_data[~st_data.obs['cell_type'].isin(['nan'])].write_h5ad('/compute/oven-0-13/jwinchel/NSCLC_ST.h5ad')